# File Operations and Change Checks

You will manage practice files and compare their contents while explaining the limits of change detection.

CSC-239 · Module 8 · Lesson 3 of 3

A developer needs to keep a saved application status, make an archive copy, and notice when a later read differs. You will first trace each file operation, then build a checker that remembers its last successful read. Every complete example creates its own temporary directory and names the files it may change.

This lesson builds on paths, UTF-8 reading and writing, String comparisons, object state, interfaces, and checked exceptions. The preceding lesson’s dictionary was a retained view of the text it loaded. Here you will distinguish such a saved observation from what a file contains later, and remove the files created by an exercise in a deliberate order.

[Review the module’s text-file vocabulary](terms.md).


## Learning Goals

- Copy, move, inspect and delete only files created by the exercise.
- Compare successive text snapshots and explain what a polling check can miss.


## Why This Matters

Applications save settings, game state, and work in progress so another run can recover them. Making a separate copy preserves one version while the working file changes. Naming and removing only the application’s own files keeps this work contained and makes the remaining data easier to understand.

Saved data can change after a program reads it. A file’s size or recorded time gives useful information, but neither is the text itself. If software treats a failed read as “unchanged,” it can keep using old state without knowing that its observation failed. Comparing successful reads and retaining the last good baseline gives the caller a clearer result.

The Ghost project combines file reading, writing, and monitoring with interfaces supplied by the instructor. This lesson connects your earlier object and exception design to those file responsibilities. It also makes one limit explicit: checking at separate moments cannot reveal every event between them.


## Check Your Starting Point

A program writes `"ready\n"` with UTF-8, then successfully reads it into a String. Explain what the Path identifies, what the returned String retains, and why a later read must use the same encoding. Which String method compares text contents? Recall why a buffered writer should finish before the read begins. Finally, explain how a method declared with `throws IOException` communicates a failed read and how a class can supply a method promised by an interface.


In [ ]:
Your response:

Path and retained String roles:


Matching encoding and String comparison:


Writer-to-reader ordering:


IOException contract and interface implementation:


<details>
<summary>Show answer</summary>

A Path identifies a location; constructing it alone does not create a file. A successful read produces a String containing the text read at that moment. Later file changes do not rewrite that immutable String. Matching UTF-8 write and read rules recovers the intended text, and `equals` compares String contents.

Finishing the buffered writer sends remaining text onward and closes it before reading begins. A declared `throws IOException` lets a failed read reach the caller’s handler instead of supplying a normal return value. A class uses `implements` and provides the interface’s required method with a compatible signature. A common mistake is using `==` for text contents or treating an exception as a successful Boolean result.

</details>


## Video Demonstration

Follow an owned status file through copying, moving, comparison, and cleanup. An explained worked case prepares you for a different text-size prediction.

<video controls preload="metadata" width="960">
<source src="media/03_file_operations_and_change_checks/demo.mp4" type="video/mp4">
<track kind="captions" src="media/03_file_operations_and_change_checks/captions.vtt" srclang="en" label="English">
Your browser does not support embedded video.
</video>

[Read the file operations and change checks demonstration transcript](media/03_file_operations_and_change_checks/transcript.md).


## Concept

### Observe the file that a path names

A developer maintains a saved application status. The program needs to make a copy, give that copy an archive name, and check whether the current status differs from an earlier observation. We will practice on files created inside a new temporary directory, so every operation has a known target belonging to this run.

A **file existence check** asks whether a path currently names an existing entry. A **file type check** asks what kind of entry it names. `Files.exists` and `Files.isRegularFile` return Boolean observations. A **regular file** stores ordinary file data; a directory instead groups entries.

The lifecycle example later uses these two expressions in its report:

```java
Files.exists(source)
```

```java
Files.isRegularFile(moved)
```

These are expression fragments, not the complete setup. `source` and `moved` are Path variables for entries created by that example. The first expression asks whether the source entry exists. The second asks whether the moved entry is a regular file. Returning `true` answers that particular question at the time of the check.

An earlier observation cannot guarantee that a later read or write will succeed. Another program could change the entry between operations. A result of `false` can also mean that Java could not determine the answer. The complete examples therefore handle `IOException` from their actual file operations instead of treating an existence check as permission to assume success.


### Copy data while keeping the original

A **file copy** creates a separate destination containing the source data while retaining the source. This is useful when an application needs a working original and another saved version. In our small lifecycle example, `source.txt` contains the text `map` and `copy.txt` is initially unused.

```java
    Files.writeString(source, "map", StandardCharsets.UTF_8);
    Files.copy(source, copy);
```

These statements come after construction of the temporary directory and its Path values. The write creates the source text. The copy then uses `source` as its input and `copy` as its destination. After successful completion, both entries contain `map`.

The names `source` and `copy` are ordinary variables holding paths; choosing those names does not perform the operation. The call to `Files.copy` creates the destination. Later changes to one regular file do not automatically keep the other file's text synchronized.

Our target begins unused because the program creates a fresh directory. With these default options, an existing destination can cause copying to fail. This lesson does not replace unrelated existing files. Each complete example reports an I/O problem if its requested operation cannot finish.

### Move the copy to its new name

A **file move** changes where an existing entry is found. Moving within one directory can rename a file. We use it to turn the newly created `copy.txt` into `moved.txt`:

```java
    Files.move(copy, moved);
```

The first argument identifies the existing copy; the second is its new destination. After this call succeeds in our fixture, the old copy path no longer names that file, and the moved path names the file containing `map`. The original `source.txt` remains because this call moves the copy, not the original.

This explains the later report: source existence is `true`, old-copy existence is `false`, and the moved entry is a regular file. We can also read its text to confirm the copied data reached the new name.

Copying and moving answer different needs. Copying leaves two files; moving gives the copied file a different location. Like the copy destination, our move destination starts unused. The steps are separate operations. If a later step fails, earlier successful changes can remain.


The complete example below isolates copying and moving before cleanup is added. It creates `source.txt` containing `map`, then gives its copied data a new name. This retained example leaves its own directory and files in temporary storage after the cell ends. The next section explains how later examples remove their owned entries.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-copy-");
    Path source = directory.resolve("source.txt");
    Path copy = directory.resolve("copy.txt");
    Path moved = directory.resolve("moved.txt");
    Files.writeString(source, "map", StandardCharsets.UTF_8);
    Files.copy(source, copy);
    Files.move(copy, moved);
    System.out.println("Source: " + Files.exists(source));
    System.out.println("Old copy: " + Files.exists(copy));
    System.out.println("Moved file: " + Files.isRegularFile(moved));
    System.out.println(Files.readString(moved, StandardCharsets.UTF_8));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Expected output:

```text
Source: true
Old copy: false
Moved file: true
map
```

The source is retained, while the copied entry is now found through `moved`. Reading that moved file recovers `map`. The four reports describe successful operations in this newly created directory; earlier success does not guarantee that every later operation will succeed.


### Remove owned files before their directory

**File deletion** removes a file entry or an empty directory entry. The method **`deleteIfExists`** returns `true` when it removed the requested entry and `false` when that entry was already absent. An I/O failure is still an exception; `false` does not stand for every possible failure.

```java
    System.out.println("Removed moved: " + Files.deleteIfExists(moved));
    System.out.println("Removed moved again: " + Files.deleteIfExists(moved));
    System.out.println("Removed source: " + Files.deleteIfExists(source));
    System.out.println("Removed directory: " + Files.deleteIfExists(directory));
```

These statements finish the lifecycle example after its reads. The first removes the moved file and prints `true`. The second repeats the request for that now-absent entry and prints `false`. The third removes the remaining source file. Only then is the directory empty and ready for removal by the fourth call.

Deleting the directory first would leave its children in the way. This method is not a recursive cleanup command. Our program names only entries it created, and the deletion order follows their relationship.

If an earlier operation throws, later statements in the try block can be skipped, including these cleanup calls. The normal-path cleanup shown here does not promise cleanup after every failure. Larger applications need a cleanup design that handles their failure paths and preserves useful information about the original problem.


This complete cleanup example creates one file containing `practice`. It then removes that file, repeats the request, and removes its now-empty directory. The second result distinguishes an already-absent entry from an I/O exception.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-clean-");
    Path file = directory.resolve("note.txt");
    Files.writeString(file, "practice", StandardCharsets.UTF_8);
    System.out.println("Removed file: " + Files.deleteIfExists(file));
    System.out.println("Removed again: " + Files.deleteIfExists(file));
    System.out.println("Removed directory: " + Files.deleteIfExists(directory));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Expected output:

```text
Removed file: true
Removed again: false
Removed directory: true
```

Only the first deletion removes the file. The repeated request has no file to remove, and the empty directory can then be removed. These three statements are ordinary normal-path operations; an earlier exception may skip later cleanup.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-lifecycle-demo-");
    Path source = directory.resolve("source.txt");
    Path copy = directory.resolve("copy.txt");
    Path moved = directory.resolve("moved.txt");
    Files.writeString(source, "map", StandardCharsets.UTF_8);
    Files.copy(source, copy);
    Files.move(copy, moved);
    System.out.println("Source: " + Files.exists(source));
    System.out.println("Old copy: " + Files.exists(copy));
    System.out.println("Moved file: " + Files.isRegularFile(moved));
    System.out.println("Moved text: " + Files.readString(moved, StandardCharsets.UTF_8));
    System.out.println("Removed moved: " + Files.deleteIfExists(moved));
    System.out.println("Removed moved again: " + Files.deleteIfExists(moved));
    System.out.println("Removed source: " + Files.deleteIfExists(source));
    System.out.println("Removed directory: " + Files.deleteIfExists(directory));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Expected output:

```text
Source: true
Old copy: false
Moved file: true
Moved text: map
Removed moved: true
Removed moved again: false
Removed source: true
Removed directory: true
```

This combines the operations just taught. The first four reports establish the separate source and moved copy and recover `map` from the moved entry. The deletion reports are true, false, true, true: remove the moved file, repeat its absent-file request, remove the source, then remove the empty directory. No other directory or file is named for removal.


<details class="animation-panel" open>
<summary>Copy, move, and remove owned files — show or hide animation</summary>
<p><img src="media/03_file_operations_and_change_checks/copy_move_delete_lifecycle.gif" alt="Writing creates source.txt containing map. Copy creates copy.txt and retains source.txt. Move changes copy.txt to moved.txt; source.txt remains separate. The observations report source true, old copy false and moved regular file true. Remove moved once, observe false for its repeated deletion, remove source, then remove the empty directory." width="960" style="max-width:100%;height:auto;"></p>
</details>

Copy retains the original file; move changes the copied file location. Deletion removes the known files before their empty directory. A repeated deletion of the already-removed moved file returns false. All targets belong to this run. The sequence repeats every 12.6 seconds. Hide the panel to remove visible motion.

[View still: Copy, move, and remove owned files](media/03_file_operations_and_change_checks/copy_move_delete_lifecycle_still.png).


### Distinguish metadata from text contents

**File metadata** describes a file apart from its text. Size and recorded modification time can help an application inspect stored data, but they answer different questions from reading and comparing the text itself.

```java
    long beforeSize = Files.size(file);
    FileTime beforeTime = Files.getLastModifiedTime(file);
```

These lines belong to the metadata example after its file has been created. **`Files.size`** returns the file's size in bytes. The keyword **`long`** names Java's wider primitive whole-number type; it matches the returned size type. We use it to store the byte count rather than narrowing the result to an int.

**`Files.getLastModifiedTime`** returns a **FileTime** object representing the file system's recorded modification time. `beforeTime` keeps that observation. FileTime is a library type, not a Java keyword, and the complete example imports it before use.

The example changes `open` to `busy`. Each of these example letters occupies one byte in UTF-8. Both strings therefore occupy four bytes, so the byte size remains four even though the text differs. A size comparison alone would miss that change.

The example also obtains a time before and after the write. Its report confirms that both timestamp objects were obtained; it does not claim the times differ. File systems have different time resolutions, so rapid writes may share the same recorded time. Metadata is useful evidence about a file, but equal size or equal recorded time does not prove unchanged contents.


The first complete metadata example records the original byte size and obtains two timestamp objects. It deliberately prints only whether the sizes match and whether both time values were obtained. Like the earlier copy-only example, this retained program leaves its private fixture in temporary storage.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.attribute.FileTime;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-meta-");
    Path file = directory.resolve("state.txt");
    Files.writeString(file, "open", StandardCharsets.UTF_8);
    long beforeSize = Files.size(file);
    FileTime beforeTime = Files.getLastModifiedTime(file);
    Files.writeString(file, "busy", StandardCharsets.UTF_8);
    FileTime afterTime = Files.getLastModifiedTime(file);
    System.out.println("Same byte size: " + (beforeSize == Files.size(file)));
    System.out.println("Timestamp observed: " + (beforeTime != null && afterTime != null));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Expected output:

```text
Same byte size: true
Timestamp observed: true
```

Both `open` and `busy` occupy four UTF-8 bytes. The second report checks that two FileTime values exist; it does not compare them. The next program also reads the actual text so the difference can be checked directly, then removes its owned fixture.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.nio.file.attribute.FileTime;
try {
    Path directory = Files.createTempDirectory("csc239-metadata-demo-");
    Path file = directory.resolve("state.txt");
    Files.writeString(file, "open", StandardCharsets.UTF_8);
    String beforeText = Files.readString(file, StandardCharsets.UTF_8);
    long beforeSize = Files.size(file);
    FileTime beforeTime = Files.getLastModifiedTime(file);
    Files.writeString(file, "busy", StandardCharsets.UTF_8);
    String afterText = Files.readString(file, StandardCharsets.UTF_8);
    FileTime afterTime = Files.getLastModifiedTime(file);
    System.out.println("Before bytes: " + beforeSize);
    System.out.println("After bytes: " + Files.size(file));
    System.out.println("Same byte size: " + (beforeSize == Files.size(file)));
    System.out.println("Timestamps obtained: " + (beforeTime != null && afterTime != null));
    System.out.println("Content changed: " + !afterText.equals(beforeText));
    Files.deleteIfExists(file);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Expected output:

```text
Before bytes: 4
After bytes: 4
Same byte size: true
Timestamps obtained: true
Content changed: true
```

The first two reports show four bytes before and after. The size comparison is true, and both timestamps were obtained. Yet the final content comparison is also true for “changed”: `open` differs from `busy`. The reports separate a byte-count observation, successful metadata retrieval, and text equality instead of treating them as interchangeable evidence.


<details class="animation-panel" open>
<summary>Compare metadata with text contents — show or hide animation</summary>
<p><img src="media/03_file_operations_and_change_checks/metadata_vs_content.gif" alt="The first successful read contains open and the stored size is four UTF-8 bytes. Record the first last-modified time as an observation T1. Rewrite busy; the new text also occupies four UTF-8 bytes. Obtain T2, which may equal T1; both timestamp objects are now available, with the confirmation report occurring later. The successful text reads differ even though their byte sizes match." width="960" style="max-width:100%;height:auto;"></p>
</details>

Both words occupy four UTF-8 bytes. The program obtains timestamps but neither prints exact time values nor promises they differ. The separate String comparison establishes that these successful reads contain different text. The sequence repeats every 12.6 seconds. Hide the panel to remove visible motion.

[View still: Compare metadata with text contents](media/03_file_operations_and_change_checks/metadata_vs_content_still.png).


### Compare a successful read before replacing the baseline

A **content snapshot** is text retained from an earlier successful read. It gives the next read something to compare against. The `StatusSnapshot` helper in the upcoming teaching example stores both the file path and a `previous` string. Its constructor reads the initial text before any check is requested.

```java
    public boolean check() throws IOException {
        String current = Files.readString(file, StandardCharsets.UTF_8);
        boolean changed = !current.equals(previous);
        previous = current;
        return changed;
    }
```

This method is part of that class, whose complete declaration follows. The first statement reads the current UTF-8 text. The next compares its contents with `previous`. `equals` returns whether they match, and **logical negation (`!`)** reverses that Boolean so `changed` means they differ.

Only after saving that comparison result does the method replace `previous` with `current`. The new snapshot becomes the **comparison baseline**, the saved reference value for the next successful check. Finally, the method returns the saved Boolean. Updating `previous` before comparing would compare the new text with itself and lose the evidence of change.

The constructor and method declare `throws IOException`. If a read inside `check()` fails, execution does not reach the assignment to `previous`, so the earlier successful snapshot remains. A failed initial constructor read instead prevents creation of a usable initialized checker. The caller receives the exception instead of a made-up unchanged result. This distinguishes a failed observation from a successful comparison returning `false`.

With an initial snapshot of `ready`, checking unchanged text returns `false`. After writing `busy`, a check returns `true` and saves `busy`. Another check without another write returns `false`. Those are three separate observations using a baseline that changes only after successful reads.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
class StatusSnapshot {
    private Path file;
    private String previous;
    public StatusSnapshot(Path file) throws IOException {
        this.file = file;
        this.previous = Files.readString(file, StandardCharsets.UTF_8);
    }
    public boolean check() throws IOException {
        String current = Files.readString(file, StandardCharsets.UTF_8);
        boolean changed = !current.equals(previous);
        previous = current;
        return changed;
    }
}

try {
    Path directory = Files.createTempDirectory("csc239-snapshot-demo-");
    Path file = directory.resolve("status.txt");
    Files.writeString(file, "ready", StandardCharsets.UTF_8);
    StatusSnapshot snapshot = new StatusSnapshot(file);
    System.out.println("Initial change: " + snapshot.check());
    Files.writeString(file, "busy", StandardCharsets.UTF_8);
    System.out.println("After rewrite: " + snapshot.check());
    System.out.println("Repeated check: " + snapshot.check());
    Files.deleteIfExists(file);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Expected output:

```text
Initial change: false
After rewrite: true
Repeated check: false
```

The constructor stores `ready`. The first call reads the same text and reports false. Rewriting `busy` makes the second successful read different, so it reports true and stores `busy`. The repeat reads `busy` again and reports false. These are three calls to the same object; each comparison uses the snapshot saved by the preceding successful read.


<details class="animation-panel" open>
<summary>Compare text before updating its snapshot — show or hide animation</summary>
<p><img src="media/03_file_operations_and_change_checks/snapshot_compare_then_commit.gif" alt="Construction saves ready; an unchanged first check compares ready with ready and returns false. After the rewrite, a successful read gives current busy while previous is still ready. Compare current busy with previous ready before changing the saved snapshot. Store busy as the snapshot only after computing changed; return the saved true result. The next successful read is busy again, so comparison returns false." width="960" style="max-width:100%;height:auto;"></p>
</details>

The constructor establishes the baseline. check reads, compares, updates and returns in that order. The same changed content is reported once because the successful check becomes the next baseline. The sequence repeats every 12.6 seconds. Hide the panel to remove visible motion.

[View still: Compare text before updating its snapshot](media/03_file_operations_and_change_checks/snapshot_compare_then_commit_still.png).


### Keep a failed observation separate from unchanged text

The next complete program deliberately removes its own status file after constructing `StatusSnapshot`. Its nested `try` attempts a check; the nested `catch` prints a fixed explanation if that read raises `IOException`. Because the call throws while evaluating the print expression, the `Unexpected result:` line is not printed.

The outer handler still covers setup, restoration, later reads, and cleanup. After the nested handler, the program recreates the same file with `ready`. Comparing that restored text with the earlier baseline tests whether the failed read left useful state intact. A later rewrite to `busy` then tests ordinary change detection again. These are caller-controlled steps, not an automatic retry or a background process.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
class StatusSnapshot {
    private Path file;
    private String previous;
    public StatusSnapshot(Path file) throws IOException {
        this.file = file;
        this.previous = Files.readString(file, StandardCharsets.UTF_8);
    }
    public boolean check() throws IOException {
        String current = Files.readString(file, StandardCharsets.UTF_8);
        boolean changed = !current.equals(previous);
        previous = current;
        return changed;
    }
}

try {
    Path directory = Files.createTempDirectory("csc239-read-failure-demo-");
    Path file = directory.resolve("status.txt");
    Files.writeString(file, "ready", StandardCharsets.UTF_8);
    StatusSnapshot snapshot = new StatusSnapshot(file);
    Files.deleteIfExists(file);
    try {
        System.out.println("Unexpected result: " + snapshot.check());
    } catch (IOException problem) {
        System.out.println("Read failed; no result returned.");
    }
    Files.writeString(file, "ready", StandardCharsets.UTF_8);
    System.out.println("After restoring ready: " + snapshot.check());
    Files.writeString(file, "busy", StandardCharsets.UTF_8);
    System.out.println("After rewriting busy: " + snapshot.check());
    Files.deleteIfExists(file);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Expected output:

```text
Read failed; no result returned.
After restoring ready: false
After rewriting busy: true
```

The failed read reaches the nested handler and supplies no Boolean result. Restoring `ready` produces false because the saved baseline is still `ready`. Writing `busy` then produces true. Returning false from the failed read would hide a different event: there was no successful new text to compare at all.


<details class="animation-panel" open>
<summary>Keep the baseline when a read fails — show or hide animation</summary>
<p><img src="media/03_file_operations_and_change_checks/failed_read_preserves_baseline.gif" alt="Construction saves ready as the last successful snapshot. The known file is removed; check reaches readString and throws IOException. No comparison, snapshot assignment or boolean return executes on the failed call. The specific caller handler reports failure instead of an unchanged result. Restore ready and observe false, then rewrite busy and observe true; the saved successful baseline survived." width="960" style="max-width:100%;height:auto;"></p>
</details>

A failed read leaves the remaining method statements unexecuted. The caller gets an exception, not false. Restoring the earlier ready content yields an unchanged result, then a later busy read detects a difference. This tests preservation instead of assuming it. The sequence repeats every 12.6 seconds. Hide the panel to remove visible motion.

[View still: Keep the baseline when a read fails](media/03_file_operations_and_change_checks/failed_read_preserves_baseline_still.png).


### Explain the gaps between checks

**Polling** means checking state at separate moments to look for changes. Our helper performs one check when the caller invokes `check`; it does not create a timer or automatically watch the file. That makes its observations and their limits visible in the program's statement order.

Suppose the saved snapshot is `ready`. The caller writes `busy`, then writes `ready` again before invoking the next check. That read sees `ready`, which equals the saved snapshot, so it returns `false`. The intermediate `busy` text existed, but this checker never read it.

In a contrasting sequence, the caller checks after writing `busy` and checks again after restoring `ready`. Both checks return `true`, because each observed value differs from the baseline saved by the preceding successful observation. The difference is where the checks occur, not whether a write took place.

A snapshot comparison therefore reports differences between successful observations. It is not a history of every write. Reading while another program writes also does not establish that the writer has finished all intended changes or that the read is a coordinated snapshot of an entire update.

The later `ChangeProbe` and `TextChangeProbe` practice applies this compare-then-update rule through a small tutorial interface. The Ghost assignment has its own supplied `FileTextReader`, `FileTextWriter`, and `AbstractFileMonitor` declarations. Use their actual signatures and communication rules when building that project; the tutorial's method names do not replace the assignment contract.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
class StatusSnapshot {
    private Path file;
    private String previous;
    public StatusSnapshot(Path file) throws IOException {
        this.file = file;
        this.previous = Files.readString(file, StandardCharsets.UTF_8);
    }
    public boolean check() throws IOException {
        String current = Files.readString(file, StandardCharsets.UTF_8);
        boolean changed = !current.equals(previous);
        previous = current;
        return changed;
    }
}

try {
    Path directory = Files.createTempDirectory("csc239-polling-demo-");
    Path file = directory.resolve("status.txt");
    Files.writeString(file, "ready", StandardCharsets.UTF_8);
    StatusSnapshot snapshot = new StatusSnapshot(file);
    Files.writeString(file, "busy", StandardCharsets.UTF_8);
    Files.writeString(file, "ready", StandardCharsets.UTF_8);
    System.out.println("One later check: " + snapshot.check());
    Files.writeString(file, "busy", StandardCharsets.UTF_8);
    System.out.println("Check after busy: " + snapshot.check());
    Files.writeString(file, "ready", StandardCharsets.UTF_8);
    System.out.println("Check after ready: " + snapshot.check());
    Files.deleteIfExists(file);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Expected output:

```text
One later check: false
Check after busy: true
Check after ready: true
```

The first report is false because both writes occur before the check: the saved and newly read text are both `ready`. In the contrast, a check follows each write. `busy` differs from saved `ready`, then restored `ready` differs from saved `busy`, so both later reports are true. The program performs the same kinds of writes; the placement of observations determines which differences it can report.


<details class="animation-panel" open>
<summary>See what happens between polling checks — show or hide animation</summary>
<p><img src="media/03_file_operations_and_change_checks/polling_unseen_intermediate_change.gif" alt="The saved successful observation is ready. The caller writes busy without checking it. The caller restores ready before the next check. One later check reads ready and reports false; the intermediate busy text was never read. In the contrast, a check after busy reports true and saves busy; a check after ready also reports true." width="960" style="max-width:100%;height:auto;"></p>
</details>

The first two writes occur between observations, so the later String equals the saved ready text. The contrast observes both transitions and updates the snapshot after each. These are explicit calls, not a background monitoring loop. The sequence repeats every 12.6 seconds. Hide the panel to remove visible motion.

[View still: See what happens between polling checks](media/03_file_operations_and_change_checks/polling_unseen_intermediate_change_still.png).


## Worked Example

### Keep an archive while inspecting the current status

A developer’s current status file is `state.txt`. Its initial text is `"ready\n"`, a status word followed by a newline. The developer wants a separate archived version, then wants to compare that earlier text with `"busy!\n"` in the working source. Both texts occupy six UTF-8 bytes: five ordinary characters and one newline. Successful reports should show a retained source, a renamed copy, equal byte size, different text, and deliberate removal of the owned entries.

The program creates its new directory first and resolves `state.txt`, `backup.txt`, and `archive.txt` inside it. These paths start unused. The complete program below recreates every value it needs; it does not depend on the preceding examples’ directories.


### Preserve one version under an archive name

These statements follow the Path declarations in the complete program:

```java
Files.writeString(state, "ready\n", StandardCharsets.UTF_8);
Files.copy(state, backup);
Files.move(backup, archive);
```

The write creates the source text. Copying creates a separate file at `backup`, and moving changes that copy’s location to `archive`. The original source remains available for the later rewrite. The existence and type reports that follow inspect those three locations at this point.


### Compare the content that was actually read

```java
String previous = Files.readString(state, StandardCharsets.UTF_8);
long previousSize = Files.size(state);
Files.writeString(state, "busy!\n", StandardCharsets.UTF_8);
String current = Files.readString(state, StandardCharsets.UTF_8);
```

The first statement keeps the earlier source text in `previous`. The next keeps its byte count in `previousSize`. The write changes the source file, and the last statement reads that new text into `current`. Rewriting the source does not synchronize the separate archive; its copied text stays `ready` followed by a newline.

The later size comparison asks whether the current byte count equals `previousSize`. The text comparison uses `!current.equals(previous)`: `equals` checks contents, and logical **`!`** reverses the Boolean so true means different. Equal six-byte sizes therefore coexist with different text. These local observations do not yet form an object that updates its baseline over many calls; that is the `StatusSnapshot` pattern taught above.


### Remove the known entries in order

The two archive deletion reports call `deleteIfExists` twice on the same path. After a successful first removal, the second has nothing left to remove. The remaining source is deleted next, and its now-empty directory is last.

All these steps sit in one outer `try` with an `IOException` handler. The handler reports the actual problem if an operation fails. Later normal-path statements may then be skipped; the program does not promise to undo earlier successful operations or finish cleanup after every failure.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-state-");
    Path state = directory.resolve("state.txt");
    Path backup = directory.resolve("backup.txt");
    Path archive = directory.resolve("archive.txt");
    Files.writeString(state, "ready\n", StandardCharsets.UTF_8);
    Files.copy(state, backup);
    Files.move(backup, archive);
    System.out.println("Source exists: " + Files.exists(state));
    System.out.println("Backup exists: " + Files.exists(backup));
    System.out.println("Archive is file: " + Files.isRegularFile(archive));
    String previous = Files.readString(state, StandardCharsets.UTF_8);
    long previousSize = Files.size(state);
    Files.writeString(state, "busy!\n", StandardCharsets.UTF_8);
    String current = Files.readString(state, StandardCharsets.UTF_8);
    System.out.println("Same size: " + (previousSize == Files.size(state)));
    System.out.println("Content changed: " + !current.equals(previous));
    System.out.println("Deleted archive: " + Files.deleteIfExists(archive));
    System.out.println("Deleted again: " + Files.deleteIfExists(archive));
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Expected output:

```text
Source exists: true
Backup exists: false
Archive is file: true
Same size: true
Content changed: true
Deleted archive: true
Deleted again: false
```

The source remains after the copy. The backup path disappears after the move, while the archive names a regular file. The two text versions have equal byte size but different content. The first archive removal returns true and the second returns false. All affected paths belong to the newly created practice directory.


## Guided Practice

### Predict another pair of status texts

Before running the next program, predict all seven reports. Its source initially holds `"idle\n"` and later holds `"playing\n"`. Track which of `state.txt`, `backup.txt`, and `archive.txt` exist after copy and after move. Count the stored bytes for these letters and newline, then predict the size and content comparisons. Explain why deleting the archive twice can give different results and why the source must be removed before its directory.


In [ ]:
Your response:

Predicted seven reports:


Paths after copy and move:


Byte counts and comparison reasoning:


Deletion order and repeated removal:


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-state-");
    Path state = directory.resolve("state.txt");
    Path backup = directory.resolve("backup.txt");
    Path archive = directory.resolve("archive.txt");
    Files.writeString(state, "idle\n", StandardCharsets.UTF_8);
    Files.copy(state, backup);
    Files.move(backup, archive);
    System.out.println("Source exists: " + Files.exists(state));
    System.out.println("Backup exists: " + Files.exists(backup));
    System.out.println("Archive is file: " + Files.isRegularFile(archive));
    String previous = Files.readString(state, StandardCharsets.UTF_8);
    long previousSize = Files.size(state);
    Files.writeString(state, "playing\n", StandardCharsets.UTF_8);
    String current = Files.readString(state, StandardCharsets.UTF_8);
    System.out.println("Same size: " + (previousSize == Files.size(state)));
    System.out.println("Content changed: " + !current.equals(previous));
    System.out.println("Deleted archive: " + Files.deleteIfExists(archive));
    System.out.println("Deleted again: " + Files.deleteIfExists(archive));
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Run the complete program. Record all seven reports and compare them with your prediction without erasing it. Account for both text sizes and deletion results. Run the whole cell again and explain why its new directory allows a fresh copy and move. Describe the files and directory removed on a successful run.


In [ ]:
Your response:

Actual seven reports:


Comparison and reasons for revisions:


Replay output and owned cleanup:


### Trace the file entries and successful reads

Trace the entries after the write, copy, move, rewrite, and cleanup. Distinguish source text from archived text. Mark where `previous` and `current` are read and which values the comparisons use. Explain why an existence observation cannot guarantee the next operation, and which later statements may be skipped after an I/O failure.


In [ ]:
Your response:

Entries and text after each operation:


Previous and current read points:


Limits of path observations:


Handler effect on later operations:


<details>
<summary>Show answer</summary>

The first three reports are true, false and true: the source remains, the backup path no longer exists after the move, and the archive is a regular file. The supplied text changes from 5 to 8 UTF-8 bytes, producing Same size false and Content changed true. Deleting the archive reports true and then false. The remaining source is removed before the empty directory. A complete successful replay creates a new directory, so it does not reuse earlier copy/move targets. Its same supplied operations produce the same seven reports.

After the first write only the source exists. Copy adds a second file with the initial text; move changes that copied file’s location from backup to archive. Rewriting the source changes its text to `playing\n`; the archive still contains the earlier copied text. The previous String was read before the rewrite and current afterward. The program compares their contents and observes the current byte size separately. Cleanup removes the archive, then the source, then the empty directory.

A path observation describes that moment. Another program could change the path before a later operation, and a false existence/type result can also mean the answer could not be determined. The actual read, copy, move or delete can still raise `IOException`. On failure, later statements in that `try`, including normal cleanup, may be skipped.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-state-");
    Path state = directory.resolve("state.txt");
    Path backup = directory.resolve("backup.txt");
    Path archive = directory.resolve("archive.txt");
    Files.writeString(state, "idle\n", StandardCharsets.UTF_8);
    Files.copy(state, backup);
    Files.move(backup, archive);
    System.out.println("Source exists: " + Files.exists(state));
    System.out.println("Backup exists: " + Files.exists(backup));
    System.out.println("Archive is file: " + Files.isRegularFile(archive));
    String previous = Files.readString(state, StandardCharsets.UTF_8);
    long previousSize = Files.size(state);
    Files.writeString(state, "playing\n", StandardCharsets.UTF_8);
    String current = Files.readString(state, StandardCharsets.UTF_8);
    System.out.println("Same size: " + (previousSize == Files.size(state)));
    System.out.println("Content changed: " + !current.equals(previous));
    System.out.println("Deleted archive: " + Files.deleteIfExists(archive));
    System.out.println("Deleted again: " + Files.deleteIfExists(archive));
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Source exists: true
Backup exists: false
Archive is file: true
Same size: false
Content changed: true
Deleted archive: true
Deleted again: false
```

Common error: Treating copy as removal of the source. Expecting the old backup path to keep naming the moved file. Ignoring the newline in these byte counts. Expecting both deletion calls to remove the same entry.

</details>


### Compare stored bytes with recovered text

The next complete program rewrites `café` as `tea` and records a FileTime before and after. Use the UTF-8 versus String-length distinction from lesson 1 and the metadata rules just taught. Predict all six reports before running. Explain what the non-null timestamp test can establish without assuming that two rapid writes receive different recorded times.


In [ ]:
Your response:

Predicted six reports:


String length versus UTF-8 byte reasoning:


Expected meaning and limit of the time report:


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.attribute.FileTime;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-metadata-");
    Path file = directory.resolve("state.txt");
    Files.writeString(file, "café", StandardCharsets.UTF_8);
    String previous = Files.readString(file, StandardCharsets.UTF_8);
    long previousSize = Files.size(file);
    FileTime beforeTime = Files.getLastModifiedTime(file);
    Files.writeString(file, "tea", StandardCharsets.UTF_8);
    String current = Files.readString(file, StandardCharsets.UTF_8);
    long currentSize = Files.size(file);
    FileTime afterTime = Files.getLastModifiedTime(file);
    System.out.println("Before characters: " + previous.length());
    System.out.println("Before bytes: " + previousSize);
    System.out.println("After bytes: " + currentSize);
    System.out.println("Times obtained: " + (beforeTime != null && afterTime != null));
    System.out.println("Same byte size: " + (previousSize == currentSize));
    System.out.println("Content changed: " + !current.equals(previous));
    Files.deleteIfExists(file);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Run the program and record its six reports. Explain any revisions, why the two measurements of `café` differ, and what each FileTime represents. State what `Times obtained:` proves and leaves unknown, and how the content comparison adds different evidence. Identify the owned cleanup order.


In [ ]:
Your response:

Actual six reports and comparison:


Text positions versus bytes:


Recorded times and limits of the report:


Content comparison and cleanup:


<details>
<summary>Show answer</summary>

The recovered `café` String has length 4 but occupies 5 UTF-8 bytes; `tea` occupies 3 bytes. The six reports are 4, 5, 3, true, false and true.

`Files.getLastModifiedTime` returns the recorded time as a `FileTime`; the program obtains one before and one after writing. The non-null check confirms both values were obtained. It does not compare the times or promise they differ. A file system may record two rapid writes with the same time. Size and recorded time are metadata, separate from the text. The String comparison establishes that these two successful reads differ. The program removes its known file before removing its empty directory.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.attribute.FileTime;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-metadata-");
    Path file = directory.resolve("state.txt");
    Files.writeString(file, "café", StandardCharsets.UTF_8);
    String previous = Files.readString(file, StandardCharsets.UTF_8);
    long previousSize = Files.size(file);
    FileTime beforeTime = Files.getLastModifiedTime(file);
    Files.writeString(file, "tea", StandardCharsets.UTF_8);
    String current = Files.readString(file, StandardCharsets.UTF_8);
    long currentSize = Files.size(file);
    FileTime afterTime = Files.getLastModifiedTime(file);
    System.out.println("Before characters: " + previous.length());
    System.out.println("Before bytes: " + previousSize);
    System.out.println("After bytes: " + currentSize);
    System.out.println("Times obtained: " + (beforeTime != null && afterTime != null));
    System.out.println("Same byte size: " + (previousSize == currentSize));
    System.out.println("Content changed: " + !current.equals(previous));
    Files.deleteIfExists(file);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Before characters: 4
Before bytes: 5
After bytes: 3
Times obtained: true
Same byte size: false
Content changed: true
```

Common error: Using String length as the stored byte size. Reading Times obtained as proof that the two times differ. Assuming a recorded time can prove that every rewrite was observed.

</details>


### Complete copy, move, inspection and deletion

The draft is incomplete and shown for editing. Replace `COPY_OPERATION`, `MOVE_OPERATION`, `TYPE_OPERATION` and `DELETE_OPERATION` with `copy`, `move`, `isRegularFile` and `deleteIfExists`, once each. Preserve the new directory, all fixed file names, the operation order and the later cleanup. Copy the completed program into the empty work cell. Reconstruct the seven reports already observed in the preceding idle/playing program. Plan the missing operations before completing the work cell; this task recalls known behavior rather than introducing a new prediction.

This sample is for repair:

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-state-");
    Path state = directory.resolve("state.txt");
    Path backup = directory.resolve("backup.txt");
    Path archive = directory.resolve("archive.txt");
    Files.writeString(state, "idle\n", StandardCharsets.UTF_8);
    Files.COPY_OPERATION(state, backup);
    Files.MOVE_OPERATION(backup, archive);
    System.out.println("Source exists: " + Files.exists(state));
    System.out.println("Backup exists: " + Files.exists(backup));
    System.out.println("Archive is file: " + Files.TYPE_OPERATION(archive));
    String previous = Files.readString(state, StandardCharsets.UTF_8);
    long previousSize = Files.size(state);
    Files.writeString(state, "playing\n", StandardCharsets.UTF_8);
    String current = Files.readString(state, StandardCharsets.UTF_8);
    System.out.println("Same size: " + (previousSize == Files.size(state)));
    System.out.println("Content changed: " + !current.equals(previous));
    System.out.println("Deleted archive: " + Files.DELETE_OPERATION(archive));
    System.out.println("Deleted again: " + Files.deleteIfExists(archive));
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```


In [ ]:
Your response:

Chosen operations and why:


Reconstructed seven reports:


Planned file and directory deletion order:


Run the completed program and record its seven reports. Compare them with the known result you reconstructed. Explain which operation retains the source, which changes the copied entry’s location, and why deletion of the directory belongs last.


In [ ]:
Your response:

Actual output and comparison:


Copy versus move effect:


Cleanup order:


<details>
<summary>Show answer</summary>

Use copy, move, isRegularFile and deleteIfExists in that order. Copy retains the source; move relocates the copied entry from backup to archive. The type check asks whether archive names a regular file at that moment. The first deletion removes archive; the supplied second deletion reports its absence. The source must then be removed before the directory is empty. The targets begin unused inside a new directory. Default copy/move can fail when a target already exists, so rerun the complete setup rather than assuming an old target will be replaced.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-state-");
    Path state = directory.resolve("state.txt");
    Path backup = directory.resolve("backup.txt");
    Path archive = directory.resolve("archive.txt");
    Files.writeString(state, "idle\n", StandardCharsets.UTF_8);
    Files.copy(state, backup);
    Files.move(backup, archive);
    System.out.println("Source exists: " + Files.exists(state));
    System.out.println("Backup exists: " + Files.exists(backup));
    System.out.println("Archive is file: " + Files.isRegularFile(archive));
    String previous = Files.readString(state, StandardCharsets.UTF_8);
    long previousSize = Files.size(state);
    Files.writeString(state, "playing\n", StandardCharsets.UTF_8);
    String current = Files.readString(state, StandardCharsets.UTF_8);
    System.out.println("Same size: " + (previousSize == Files.size(state)));
    System.out.println("Content changed: " + !current.equals(previous));
    System.out.println("Deleted archive: " + Files.deleteIfExists(archive));
    System.out.println("Deleted again: " + Files.deleteIfExists(archive));
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Source exists: true
Backup exists: false
Archive is file: true
Same size: false
Content changed: true
Deleted archive: true
Deleted again: false
```

Common error: Swapping source and destination arguments while filling a method name. Replacing a type observation with a file-changing operation. Deleting the directory before its known files. Assuming default copy or move silently replaces an existing target.

</details>


### Separate a write from a content change

Change only the later write in the working starter so it writes `"idle\n"` again. Keep the initial text and all file operations unchanged. Plan the edit and predict the size and content reports before running the complete modified cell.


In [ ]:
Your response:

Exact planned change:


Predicted size and content reports:


Why a write and a changed observation differ:


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-state-");
    Path state = directory.resolve("state.txt");
    Path backup = directory.resolve("backup.txt");
    Path archive = directory.resolve("archive.txt");
    Files.writeString(state, "idle\n", StandardCharsets.UTF_8);
    Files.copy(state, backup);
    Files.move(backup, archive);
    System.out.println("Source exists: " + Files.exists(state));
    System.out.println("Backup exists: " + Files.exists(backup));
    System.out.println("Archive is file: " + Files.isRegularFile(archive));
    String previous = Files.readString(state, StandardCharsets.UTF_8);
    long previousSize = Files.size(state);
    Files.writeString(state, "playing\n", StandardCharsets.UTF_8);
    String current = Files.readString(state, StandardCharsets.UTF_8);
    System.out.println("Same size: " + (previousSize == Files.size(state)));
    System.out.println("Content changed: " + !current.equals(previous));
    System.out.println("Deleted archive: " + Files.deleteIfExists(archive));
    System.out.println("Deleted again: " + Files.deleteIfExists(archive));
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Run your same-content version. Record all reports and compare the size and content results with your prediction. Explain what was written and what the two successful reads establish.


In [ ]:
Your response:

Actual reports and comparison:


Write event versus observed text difference:


For a second complete version, rewrite the source as `"busy\n"` instead. Keep the initial `"idle\n"` and every other statement unchanged. Predict the size and content reports, then build this version in the next Java cell.


In [ ]:
Your response:

Predicted size and content reports:


Stored byte and text comparison reasoning:


Run the second version and record its output. Compare its two comparison reports with the same-content version. Explain why equal byte size cannot establish equal text.


In [ ]:
Your response:

Actual output and comparison:


Why equal byte size is insufficient:


<details>
<summary>Show answer</summary>

Rewriting `idle\n` leaves the recovered content and its 5-byte size unchanged, so the reports are Same size true and Content changed false. A write occurred, but the two snapshots are equal.

Rewriting as `busy\n` also uses 5 bytes, yet the text differs from `idle\n`; that case reports true for both Same size and Content changed. Equal byte size alone cannot establish equal text. Copy, move and cleanup still operate on only this run’s new files.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-state-");
    Path state = directory.resolve("state.txt");
    Path backup = directory.resolve("backup.txt");
    Path archive = directory.resolve("archive.txt");
    Files.writeString(state, "idle\n", StandardCharsets.UTF_8);
    Files.copy(state, backup);
    Files.move(backup, archive);
    System.out.println("Source exists: " + Files.exists(state));
    System.out.println("Backup exists: " + Files.exists(backup));
    System.out.println("Archive is file: " + Files.isRegularFile(archive));
    String previous = Files.readString(state, StandardCharsets.UTF_8);
    long previousSize = Files.size(state);
    Files.writeString(state, "idle\n", StandardCharsets.UTF_8);
    String current = Files.readString(state, StandardCharsets.UTF_8);
    System.out.println("Same size: " + (previousSize == Files.size(state)));
    System.out.println("Content changed: " + !current.equals(previous));
    System.out.println("Deleted archive: " + Files.deleteIfExists(archive));
    System.out.println("Deleted again: " + Files.deleteIfExists(archive));
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Source exists: true
Backup exists: false
Archive is file: true
Same size: true
Content changed: false
Deleted archive: true
Deleted again: false
```

Common error: Returning changed merely because a write statement ran. Treating equal byte sizes as equal content. Changing the initial snapshot as well as the later write, making a different test.

**Additional test: `Rewrite idle and a newline as busy and a newline`.** Both Strings occupy 5 UTF-8 bytes, but they differ. This complete program demonstrates true for Same size and true for Content changed.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-state-");
    Path state = directory.resolve("state.txt");
    Path backup = directory.resolve("backup.txt");
    Path archive = directory.resolve("archive.txt");
    Files.writeString(state, "idle\n", StandardCharsets.UTF_8);
    Files.copy(state, backup);
    Files.move(backup, archive);
    System.out.println("Source exists: " + Files.exists(state));
    System.out.println("Backup exists: " + Files.exists(backup));
    System.out.println("Archive is file: " + Files.isRegularFile(archive));
    String previous = Files.readString(state, StandardCharsets.UTF_8);
    long previousSize = Files.size(state);
    Files.writeString(state, "busy\n", StandardCharsets.UTF_8);
    String current = Files.readString(state, StandardCharsets.UTF_8);
    System.out.println("Same size: " + (previousSize == Files.size(state)));
    System.out.println("Content changed: " + !current.equals(previous));
    System.out.println("Deleted archive: " + Files.deleteIfExists(archive));
    System.out.println("Deleted again: " + Files.deleteIfExists(archive));
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Source exists: true
Backup exists: false
Archive is file: true
Same size: true
Content changed: true
Deleted archive: true
Deleted again: false
```

</details>


### Repair snapshot update order

The faulty `SnapshotCheck` should report whether the current read differs from the previous successful read. Predict its three reports when the file changes from `cold` to `warm`. Trace `previous`, `current` and `changed` inside the faulty method. Reorder only the snapshot assignment and comparison so the comparison uses the earlier snapshot. Plan the repair and expected repaired output before placing your complete repaired program in the empty Java work cell.

This sample is for repair:

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
class SnapshotCheck {
    private Path file;
    private String previous;
    public SnapshotCheck(Path file) throws IOException {
        this.file = file;
        previous = Files.readString(file, StandardCharsets.UTF_8);
    }
    public boolean check() throws IOException {
        String current = Files.readString(file, StandardCharsets.UTF_8);
        previous = current;
        boolean changed = !current.equals(previous);
        return changed;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-snapshot-");
    Path state = directory.resolve("state.txt");
    Files.writeString(state, "cold", StandardCharsets.UTF_8);
    SnapshotCheck checker = new SnapshotCheck(state);
    System.out.println("Initial: " + checker.check());
    Files.writeString(state, "warm", StandardCharsets.UTF_8);
    System.out.println("After rewrite: " + checker.check());
    System.out.println("Repeat: " + checker.check());
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```


In [ ]:
Your response:

Predicted faulty three reports:


Faulty previous, current and changed trace:


Planned repair and expected repaired reports:


Run the repaired program. Record the three results and explain why the different text is reported once, while the repeated check is unchanged. Trace the saved baseline before and after each successful check, and compare the result with your planned repair.


In [ ]:
Your response:

Actual repaired output and comparison:


Baseline before and after each check:


Reason the repeat is unchanged:


Keep the repaired checker. In a second complete version, change only the later text from `warm` to `cold`. Predict the three results before building and running that version in the next work cell.


In [ ]:
Your response:

Predicted same-content reports:


Expected baseline and comparison sequence:


Run the same-content version and record its reports. Explain why executing a write does not itself require a true result. State why the method must still save the successful snapshot after comparing.


In [ ]:
Your response:

Actual same-content output and comparison:


Why a write alone does not imply change:


Why compare-then-update is still required:


<details>
<summary>Show answer</summary>

The faulty method sets previous to current before comparing them, so both names describe equal text and every result is false. Compute changed first, then store current as the snapshot for the next successful check.

The initial cold read is unchanged, warm differs from cold, and the next warm read matches the updated snapshot: false, true, false. The same-content test reports false three times. Updating the snapshot after the comparison is still necessary; otherwise repeated checks would keep comparing against the constructor’s old text.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
class SnapshotCheck {
    private Path file;
    private String previous;
    public SnapshotCheck(Path file) throws IOException {
        this.file = file;
        previous = Files.readString(file, StandardCharsets.UTF_8);
    }
    public boolean check() throws IOException {
        String current = Files.readString(file, StandardCharsets.UTF_8);
        boolean changed = !current.equals(previous);
        previous = current;
        return changed;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-snapshot-");
    Path state = directory.resolve("state.txt");
    Files.writeString(state, "cold", StandardCharsets.UTF_8);
    SnapshotCheck checker = new SnapshotCheck(state);
    System.out.println("Initial: " + checker.check());
    Files.writeString(state, "warm", StandardCharsets.UTF_8);
    System.out.println("After rewrite: " + checker.check());
    System.out.println("Repeat: " + checker.check());
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Initial: false
After rewrite: true
Repeat: false
```

Common error: Always returning true after a read instead of comparing text. Removing the snapshot update entirely. Keeping the assignment before the comparison. Expecting another change on a repeat read with no rewrite.

**Additional test: `Repaired checker with cold rewritten as cold`.** All three observations are unchanged. This checks that the repair detects text differences, not the fact that a write statement was executed.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
class SnapshotCheck {
    private Path file;
    private String previous;
    public SnapshotCheck(Path file) throws IOException {
        this.file = file;
        previous = Files.readString(file, StandardCharsets.UTF_8);
    }
    public boolean check() throws IOException {
        String current = Files.readString(file, StandardCharsets.UTF_8);
        boolean changed = !current.equals(previous);
        previous = current;
        return changed;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-snapshot-");
    Path state = directory.resolve("state.txt");
    Files.writeString(state, "cold", StandardCharsets.UTF_8);
    SnapshotCheck checker = new SnapshotCheck(state);
    System.out.println("Initial: " + checker.check());
    Files.writeString(state, "cold", StandardCharsets.UTF_8);
    System.out.println("After rewrite: " + checker.check());
    System.out.println("Repeat: " + checker.check());
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Initial: false
After rewrite: false
Repeat: false
```

</details>


## Independent Practice

### Construct the tutorial text change checker

Define interface `ChangeProbe` with `boolean check() throws IOException`. Implement `TextChangeProbe` with a constructor that receives a `Path`, stores it, and reads the initial UTF-8 text as its snapshot. Each successful check must read current text, compare it with the previous snapshot, store the current snapshot, and return whether the two reads differ. Declare `IOException` so the caller can distinguish a failed read from an unchanged observation.

Create `state.txt` with text `open` in a new temporary directory using prefix `csc239-probe-`. Construct a `TextChangeProbe` and use it through a `ChangeProbe` variable. Check once, rewrite `busy` and check, check again without rewriting, then rewrite `""` and check. Print the results with `Initial change: `, `After rewrite: `, `Without rewrite: ` and `After empty: ` in that order. Include all imports and an outer `IOException` handler that prints `File problem: ` plus the message. Delete only this example’s state file and then its empty directory. Before writing and running your program, predict the four results and plan the interface, object state, read/compare/update order, exception contract, and owned cleanup.

These tutorial types do not replace the instructor’s `AbstractFileMonitor` or `FileManager` declarations.


In [ ]:
Your response:

Predicted four reports:


Interface and implementation responsibilities:


Constructor and check state plan:


Exception and cleanup plan:


Run your complete checker. Record the four reports and compare them with your predictions. Trace the snapshot before and after each successful check, explain why empty text is a successful read, and distinguish a failed read from a returned false result. Identify the file and directory cleanup order.


In [ ]:
Your response:

Actual four reports and comparison:


Snapshot before and after each successful check:


Empty text versus failed read:


Owned cleanup order:


<details>
<summary>Show answer</summary>

The constructor captures open. The first check reads open and returns false. The busy read differs, so it returns true and stores busy. The next busy read matches and returns false. Reading empty text differs from busy, so the final result is true and the snapshot becomes empty.

The method stores a new snapshot only after a successful read and comparison. The interface and implementation declare IOException, leaving failure handling with the caller. Successful cleanup removes the known file before its empty directory. The original four reports provide the baseline below.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
interface ChangeProbe {
    boolean check() throws IOException;
}
class TextChangeProbe implements ChangeProbe {
    private Path file;
    private String previous;
    public TextChangeProbe(Path file) throws IOException {
        this.file = file;
        previous = Files.readString(file, StandardCharsets.UTF_8);
    }
    @Override
    public boolean check() throws IOException {
        String current = Files.readString(file, StandardCharsets.UTF_8);
        boolean changed = !current.equals(previous);
        previous = current;
        return changed;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-probe-");
    Path state = directory.resolve("state.txt");
    Files.writeString(state, "open", StandardCharsets.UTF_8);
    ChangeProbe probe = new TextChangeProbe(state);
    System.out.println("Initial change: " + probe.check());
    Files.writeString(state, "busy", StandardCharsets.UTF_8);
    System.out.println("After rewrite: " + probe.check());
    System.out.println("Without rewrite: " + probe.check());
    Files.writeString(state, "", StandardCharsets.UTF_8);
    System.out.println("After empty: " + probe.check());
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Initial change: false
After rewrite: true
Without rewrite: false
After empty: true
```

Common error: Comparing every check only with the constructor’s original text. Updating the snapshot before computing the result. Catching a failed read and returning false as though it had succeeded. Removing a directory before its file.

</details>


### Test unchanged writes, failed reads and missed changes

Keep your `ChangeProbe` and `TextChangeProbe` implementations unchanged. Run each case as a complete program with a new directory. Predict the labeled results for all four cases before the first run. Run all four complete programs in order, then record the actual results and explain each comparison by case.

1. In the original sequence, replace only the `busy` write with another `open` write. Keep the later empty write.
2. Create `open`, construct the probe, then remove the state file before calling `check`. Put that call in a nested `try` that would print `Unexpected result: ` plus the result if it returned. Catch `IOException` there and print `Missing file reported.`. Recreate the same state path with its original text `open`, then print `After recovery: ` with another check. Next write `busy` and print `After rewrite: ` with a further check. Clean up the recreated file and its directory. Explain how restoring the original text tests whether the earlier successful snapshot survived the failed read.
3. Create text `A` and construct a new probe. Write `B` and then `A` without a check between them. Print `After A-B-A: ` with one check, then clean up.
4. Repeat the A/B/A setup, but check after writing B and again after writing A. Print `After B: ` and `After A: `, then clean up.


In [ ]:
Your response:

Same-content rewrite and later empty text:


Removed file, original-content recovery and later different text:


A/B/A with one later check:


Checks after both B and A:


After recording all four predictions, use the Java work area for all four complete case versions in the listed order. Each version creates its own directory and file. Preserve each run’s labeled output, then record the observations together after the last version. Keep the interface and checker unchanged. The intentionally blank work cell is not a completed test program.


After running all four complete cases, record their actual reports and snapshot explanations in the named spaces. Compare each result with its prediction. Explain why the two A/B/A cases differ, what restoring original text checks after a failed read, and what a successful false result does and does not establish about writes between observations.


In [ ]:
Your response:

Same-content rewrite and later empty text:


Removed file, original-content recovery and later different text:


A/B/A with one later check:


Checks after both B and A:


Meaning and limits of a successful false result:


<details>
<summary>Show answer</summary>

Rewriting open as open reports no change, while the later empty text still differs.

Removing the file makes readString throw IOException, so the missing-file test reports a failure instead of a boolean result. The previous successful snapshot should remain open. Restoring open must therefore report false; this checks preservation directly. Writing busy afterward must report true, confirming that a later different read is still detected.

In the unobserved A/B/A case, the next read is A, equal to the saved A, so the result is false even though B was written. When checks occur after both writes, B differs from A and then A differs from B, so both results are true. Polling compares separate successful reads; it cannot recover an intermediate value that was never read.

**Additional test: Rewrite open as open before the empty-text step.** The same-content rewrite is unchanged. The repeat also stays unchanged; empty text later differs from open.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
interface ChangeProbe {
    boolean check() throws IOException;
}
class TextChangeProbe implements ChangeProbe {
    private Path file;
    private String previous;
    public TextChangeProbe(Path file) throws IOException {
        this.file = file;
        previous = Files.readString(file, StandardCharsets.UTF_8);
    }
    @Override
    public boolean check() throws IOException {
        String current = Files.readString(file, StandardCharsets.UTF_8);
        boolean changed = !current.equals(previous);
        previous = current;
        return changed;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-probe-");
    Path state = directory.resolve("state.txt");
    Files.writeString(state, "open", StandardCharsets.UTF_8);
    ChangeProbe probe = new TextChangeProbe(state);
    System.out.println("Initial change: " + probe.check());
    Files.writeString(state, "open", StandardCharsets.UTF_8);
    System.out.println("After rewrite: " + probe.check());
    System.out.println("Without rewrite: " + probe.check());
    Files.writeString(state, "", StandardCharsets.UTF_8);
    System.out.println("After empty: " + probe.check());
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Initial change: false
After rewrite: false
Without rewrite: false
After empty: true
```

**Additional test: Remove the file, restore open, then change it to busy.** The failed read returns no boolean. Restoring the original open text must report unchanged, directly checking that the last successful snapshot survived. The following busy rewrite must report a change. The unexpected-result line must not appear.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
interface ChangeProbe {
    boolean check() throws IOException;
}
class TextChangeProbe implements ChangeProbe {
    private Path file;
    private String previous;
    public TextChangeProbe(Path file) throws IOException {
        this.file = file;
        previous = Files.readString(file, StandardCharsets.UTF_8);
    }
    @Override
    public boolean check() throws IOException {
        String current = Files.readString(file, StandardCharsets.UTF_8);
        boolean changed = !current.equals(previous);
        previous = current;
        return changed;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-probe-");
    Path state = directory.resolve("state.txt");
    Files.writeString(state, "open", StandardCharsets.UTF_8);
    ChangeProbe probe = new TextChangeProbe(state);
    Files.deleteIfExists(state);
    try {
        System.out.println("Unexpected result: " + probe.check());
    } catch (IOException problem) {
        System.out.println("Missing file reported.");
    }
    Files.writeString(state, "open", StandardCharsets.UTF_8);
    System.out.println("After recovery: " + probe.check());
    Files.writeString(state, "busy", StandardCharsets.UTF_8);
    System.out.println("After rewrite: " + probe.check());
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Missing file reported.
After recovery: false
After rewrite: true
```

**Additional test: Write B and return to A before checking.** The only new observation is A, equal to the saved A. The program really performs both writes, but this checker never reads B.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
interface ChangeProbe {
    boolean check() throws IOException;
}
class TextChangeProbe implements ChangeProbe {
    private Path file;
    private String previous;
    public TextChangeProbe(Path file) throws IOException {
        this.file = file;
        previous = Files.readString(file, StandardCharsets.UTF_8);
    }
    @Override
    public boolean check() throws IOException {
        String current = Files.readString(file, StandardCharsets.UTF_8);
        boolean changed = !current.equals(previous);
        previous = current;
        return changed;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-probe-");
    Path state = directory.resolve("state.txt");
    Files.writeString(state, "A", StandardCharsets.UTF_8);
    ChangeProbe probe = new TextChangeProbe(state);
    Files.writeString(state, "B", StandardCharsets.UTF_8);
    Files.writeString(state, "A", StandardCharsets.UTF_8);
    System.out.println("After A-B-A: " + probe.check());
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
After A-B-A: false
```

**Additional test: Check after B and after the return to A.** The first successful check stores B after finding a difference from A. The next check compares A with B, so both observed changes are reported.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
interface ChangeProbe {
    boolean check() throws IOException;
}
class TextChangeProbe implements ChangeProbe {
    private Path file;
    private String previous;
    public TextChangeProbe(Path file) throws IOException {
        this.file = file;
        previous = Files.readString(file, StandardCharsets.UTF_8);
    }
    @Override
    public boolean check() throws IOException {
        String current = Files.readString(file, StandardCharsets.UTF_8);
        boolean changed = !current.equals(previous);
        previous = current;
        return changed;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-probe-");
    Path state = directory.resolve("state.txt");
    Files.writeString(state, "A", StandardCharsets.UTF_8);
    ChangeProbe probe = new TextChangeProbe(state);
    Files.writeString(state, "B", StandardCharsets.UTF_8);
    System.out.println("After B: " + probe.check());
    Files.writeString(state, "A", StandardCharsets.UTF_8);
    System.out.println("After A: " + probe.check());
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
After B: true
After A: true
```

</details>


## Summary

Existence and type checks describe observations at one moment. Copying retains a source and creates another file; moving changes an existing entry’s location. Remove only owned files, then their empty directory. Repeating `deleteIfExists` on an already-removed entry returns false; other failures can still throw.

Size measures bytes, and FileTime represents recorded time. Neither is the text itself. A snapshot checker reads successfully, compares with its previous text, stores the new baseline, and returns the saved comparison. A failed read supplies no new comparison and must leave the earlier baseline intact.

Polling compares separate successful observations. It can detect differing text at those moments, but it can miss an intermediate change that was overwritten before the next read.


Close the answers. Reconstruct the difference between copy and move and explain the file-before-directory deletion order. Then describe a same-size text change and a sequence that polling misses. State the required order inside a snapshot check and what happens to the baseline after a failed read.


In [ ]:
Your response:

Copy, move and deletion order:


Same-size change and missed intermediate text:


Read, compare, update and return order:


Baseline after a failed read:


<details>
<summary>Show answer</summary>

Copying retains its source while creating another file; moving changes the location of the moved entry. A directory must be empty before this deletion method removes it. For ASCII UTF-8 text, `open` and `busy` have the same four-byte size but different contents.

With baseline `ready`, writes to `busy` and back to `ready` before another check leave no observed difference. A check must read, save the comparison with the prior baseline, update the baseline, and return the saved result. If the read throws, later statements are not reached and the earlier baseline remains. A common mistake is claiming that false proves no write occurred or treating a failed read as false.

</details>


## Reflection

A game-state file may change between two reads. Describe the evidence your player has after a successful change check and what remains unknown. Explain how the caller should distinguish an I/O failure from no observed change, and give a concrete sequence of writes and checks that shows the distinction between file history and observed text.


In [ ]:
Your response:

Evidence after a successful check:


What remains unknown:


Failure versus unchanged observation:


Concrete writes and checks sequence:


Module 9 uses the file and dictionary mechanisms in the Ghost project. Its project guide and Canvas pages provide milestones and discussion; no notebook is required. Carry forward the distinction between saved object state and later file contents, and follow the supplied interfaces and communication rules when connecting those pieces.


## Supplemental Reading

- [Java 21 Files API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/nio/file/Files.html) documents copy, move, delete, checks, metadata, and text reads.
- [Java 21 FileTime API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/nio/file/attribute/FileTime.html) describes file timestamps and comparisons.
- [Java 21 Path API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/nio/file/Path.html) explains the locations used by file operations.
